# Synthetic Data Generation

In [1]:
import numpy as np
import pandas as pd

# Reproducibility
rng = np.random.default_rng(42)

# Synthetic dataset with 10 rows
data = {
    "application_id": [f"SYN-{i:04d}" for i in range(1, 11)],

    # LightGBM input features
    "annual_income": [
        92000, 68000, 54000, 125000, 47000,
        76000, 61000, 103000, 39000, 85000
    ],
    "debt_to_income": [
        0.22, 0.31, 0.46, 0.19, 0.39,
        0.28, 0.35, 0.24, 0.52, 0.27
    ],
    "interest_rate": [
        7.10, 8.25, 10.90, 6.85, 9.75,
        7.85, 8.95, 7.35, 12.40, 7.60
    ],

    # Synthetic audit-only demographic and geographic proxies
    "age_bracket": [
        "35-44", "25-34", "45-54", "55-64", "25-34",
        "35-44", "65+", "45-54", "25-34", "35-44"
    ],
    "minority_status_probability": [
        0.18, 0.74, 0.81, 0.11, 0.66,
        0.57, 0.22, 0.79, 0.88, 0.14
    ],
    "simulated_zip_redlining_proxy": [
        0.12, 0.68, 0.77, 0.09, 0.59,
        0.41, 0.33, 0.72, 0.91, 0.18
    ],
    "simulated_protected_group": [
        "reference", "comparison", "comparison", "reference",
        "comparison", "comparison", "reference", "comparison",
        "comparison", "reference"
    ],

    # Synthetic model outcomes
    "credit_decision": [
        "approved", "approved", "denied", "approved", "denied",
        "approved", "denied", "approved", "denied", "approved"
    ],
    "approval_probability": [
        0.91, 0.62, 0.34, 0.96, 0.44,
        0.71, 0.48, 0.66, 0.21, 0.88
    ],
    "interest_rate_offered": [
        7.10, 8.25, 10.90, 6.85, 9.75,
        7.85, 8.95, 7.35, 12.40, 7.60
    ]
}

df = pd.DataFrame(data)

# Save the synthetic dataset
df.to_csv("../data/synthetic_ecoa_stress_test.csv", index=False)

print(df)

  application_id  annual_income  debt_to_income  interest_rate age_bracket  \
0       SYN-0001          92000            0.22           7.10       35-44   
1       SYN-0002          68000            0.31           8.25       25-34   
2       SYN-0003          54000            0.46          10.90       45-54   
3       SYN-0004         125000            0.19           6.85       55-64   
4       SYN-0005          47000            0.39           9.75       25-34   
5       SYN-0006          76000            0.28           7.85       35-44   
6       SYN-0007          61000            0.35           8.95         65+   
7       SYN-0008         103000            0.24           7.35       45-54   
8       SYN-0009          39000            0.52          12.40       25-34   
9       SYN-0010          85000            0.27           7.60       35-44   

   minority_status_probability  simulated_zip_redlining_proxy  \
0                         0.18                           0.12   
1          

# generate the data programmatically
The following version generates a larger synthetic dataset while preserving the same schema and approximate relationships:

In [3]:
rng = np.random.default_rng(42)
n = 1000

# Synthetic audit group
protected_group = rng.choice(
    ["reference", "comparison"],
    size=n,
    p=[0.5, 0.5]
)

# Synthetic features
annual_income = np.round(
    np.clip(rng.lognormal(mean=np.log(70000), sigma=0.45, size=n), 25000, 250000),
    0
).astype(int)

debt_to_income = np.round(
    np.clip(rng.beta(a=3, b=7, size=n) * 0.85, 0.05, 0.85),
    3
)

interest_rate = np.round(
    np.clip(
        5.5
        + 9.0 * debt_to_income
        + rng.normal(0, 1.0, size=n),
        5.0,
        18.0
    ),
    2
)

age_bracket = rng.choice(
    ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    size=n,
    p=[0.08, 0.22, 0.25, 0.22, 0.15, 0.08]
)

# Synthetic probability proxy, not an observed demographic attribute
minority_status_probability = np.round(
    np.clip(
        np.where(
            protected_group == "comparison",
            rng.normal(0.72, 0.15, size=n),
            rng.normal(0.20, 0.12, size=n)
        ),
        0,
        1
    ),
    3
)

# Synthetic neighborhood-level proxy
simulated_zip_redlining_proxy = np.round(
    np.clip(
        0.2
        + 0.55 * minority_status_probability
        + rng.normal(0, 0.12, size=n),
        0,
        1
    ),
    3
)

# Synthetic approval score based on legitimate credit variables
raw_score = (
    0.000012 * annual_income
    - 1.8 * debt_to_income
    - 0.075 * interest_rate
    + rng.normal(0, 0.18, size=n)
)

approval_probability = 1 / (1 + np.exp(-raw_score))
approval_probability = np.round(approval_probability, 4)

credit_decision = np.where(
    approval_probability >= 0.50,
    "approved",
    "denied"
)

interest_rate_offered = np.round(interest_rate, 2)

df_generated = pd.DataFrame({
    "application_id": [f"SYN-{i:04d}" for i in range(1, n + 1)],
    "annual_income": annual_income,
    "debt_to_income": debt_to_income,
    "interest_rate": interest_rate,
    "age_bracket": age_bracket,
    "minority_status_probability": minority_status_probability,
    "simulated_zip_redlining_proxy": simulated_zip_redlining_proxy,
    "simulated_protected_group": protected_group,
    "credit_decision": credit_decision,
    "approval_probability": approval_probability,
    "interest_rate_offered": interest_rate_offered
})

df_generated.to_csv("../data/synthetic_ecoa_stress_test_1000_rows.csv", index=False)

print(df_generated.head())
print(df_generated.shape)


  application_id  annual_income  debt_to_income  interest_rate age_bracket  \
0       SYN-0001          50124           0.187           7.43       55-64   
1       SYN-0002         106108           0.131           7.29       35-44   
2       SYN-0003          71099           0.309           6.44       45-54   
3       SYN-0004          61635           0.340           7.08       25-34   
4       SYN-0005          66734           0.235           8.28       45-54   

   minority_status_probability  simulated_zip_redlining_proxy  \
0                        0.887                          0.927   
1                        0.307                          0.256   
2                        0.469                          0.526   
3                        0.667                          0.671   
4                        0.283                          0.447   

  simulated_protected_group credit_decision  approval_probability  \
0                comparison          denied                0.4396   
1 

# Basic disparate-impact calculation

In [4]:
# Convert the decision to a binary approval indicator
df_generated["approved"] = (
    df_generated["credit_decision"] == "approved"
).astype(int)

# Selection rate by synthetic group
selection_rates = (
    df_generated
    .groupby("simulated_protected_group")["approved"]
    .mean()
)

reference_rate = selection_rates.get("reference", np.nan)
comparison_rate = selection_rates.get("comparison", np.nan)

adverse_impact_ratio = comparison_rate / reference_rate
approval_rate_difference = comparison_rate - reference_rate

print("Selection rates:")
print(selection_rates)

print(f"\nAdverse impact ratio: {adverse_impact_ratio:.4f}")
print(f"Approval-rate difference: {approval_rate_difference:.4f}")

Selection rates:
simulated_protected_group
comparison    0.386318
reference     0.330020
Name: approved, dtype: float64

Adverse impact ratio: 1.1706
Approval-rate difference: 0.0563


For a LightGBM model, use only:

In [5]:
model_features = [
    "annual_income",
    "debt_to_income",
    "interest_rate"
]

X = df_generated[model_features]
y = df_generated["approved"]